# 6 · Pretrained classifier experiments

Reuse existing labels without changing the detector, original data, or earlier runs.
Train two separately named MobileNetV2 candidates, then compare all classifiers on
identical saved detector boxes and the previously checked development pairs.
The frozen final test split stays unused.

Use a Colab GPU runtime. Save a private copy in Drive to retain executed results.
This public notebook intentionally has no saved outputs. Keep raw logs, predictions,
images, labels, weights and executed notebooks in Drive; publish only reviewed aggregates.

In [ ]:
"""Embedded in each notebook's setup cell; uses only the standard library."""
import os
import subprocess
import sys
from importlib.metadata import PackageNotFoundError, version as package_version


NUMPY_REQUIREMENT = "numpy==2.1.3" if sys.version_info >= (3, 13) else "numpy==1.26.4"
TENSORFLOW_REQUIREMENT = "tensorflow==2.20.0" if sys.version_info >= (3, 13) else "tensorflow==2.19.1"
KERAS_REQUIREMENT = "keras==3.10.0" if sys.version_info >= (3, 13) else "keras==3.9.2"


def install_packages(requirements):
    if os.environ.get("DOMINO_SKIP_INSTALL") == "1":
        return
    if not (3, 10) <= sys.version_info[:2] <= (3, 13):
        raise RuntimeError("Use a Python 3.10–3.13 runtime for these pinned packages.")
    if sys.platform == "darwin" and sys.version_info >= (3, 13):
        raise RuntimeError("Use Colab for Python 3.13, or Python 3.12 for local macOS runs.")
    restart_message = "Choose Runtime → Restart session, then rerun setup before continuing."
    if globals().get("_DOMINO_RESTART_REQUIRED"):
        raise RuntimeError(restart_message)

    def installed(name):
        try:
            return package_version(name)
        except PackageNotFoundError:
            return None

    modules = {"numpy": "numpy", "pandas": "pandas", "matplotlib": "matplotlib",
               "pillow": "PIL", "opencv-python": "cv2", "torch": "torch",
               "torchvision": "torchvision", "tensorflow": "tensorflow", "keras": "keras",
               "scipy": "scipy", "scikit-learn": "sklearn", "h5py": "h5py",
               "jax": "jax", "jaxlib": "jaxlib", "ml-dtypes": "ml_dtypes",
               "litert-torch": "litert_torch", "torchao": "torchao"}
    loaded = {name: installed(name) for name, module in modules.items() if module in sys.modules}
    print("Installing packages for Python", sys.version.split()[0], "— progress follows:", flush=True)
    command = [sys.executable, "-u", "-m", "pip", "install",
               "--only-binary=numpy,pandas", *requirements]
    process = subprocess.Popen(command, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    try:
        for line in process.stdout:
            print(line, end="", flush=True)
        returncode = process.wait()
    except KeyboardInterrupt:
        process.terminate()
        try:
            process.wait(timeout=5)
        except subprocess.TimeoutExpired:
            process.kill()
            process.wait()
        raise
    finally:
        process.stdout.close()

    # Colab can preload NumPy. Changing files on disk cannot replace its loaded extensions.
    changed = [name for name, previous in loaded.items() if installed(name) != previous]
    if changed:
        globals()["_DOMINO_RESTART_REQUIRED"] = True
        print("Loaded packages changed:", ", ".join(changed), flush=True)
    if returncode:
        raise RuntimeError("Package installation failed. Read the pip error above before continuing. "
                           + (restart_message if changed else ""))
    if changed:
        raise RuntimeError("Installation finished. " + restart_message)
    print("Installation finished. Continue to configuration.", flush=True)


install_packages([TENSORFLOW_REQUIREMENT, KERAS_REQUIREMENT, NUMPY_REQUIREMENT,
                  "pandas==2.2.3", "matplotlib==3.10.1", "pillow==11.1.0",
                  "opencv-python==4.11.0.86"])

## Configuration and immutable code
A new experiment requires new run names and a new comparison name. Existing run
folders are never overwritten. The code revision below is fixed for reproducibility.
If a job is interrupted, its best/last checkpoints remain in that run directory.
Do not rename old folders or delete labels to restart a trial.

In [ ]:
from pathlib import Path
import json
import os
import subprocess
import sys
import urllib.request

REVISION = "fb897b1ce6598b9e57970a6ec22719d11136a702"
PROJECT = Path("/content/drive/MyDrive/DominoesCalculator")
CODE = Path("/content/domino-classifier-v3") / REVISION
TRIALS = [("pips-v3-mnv2-rgb-01", "rgb"), ("pips-v3-mnv2-mixed-01", "mixed")]
COMPARISON = "classifier-v3-01"
FILES = ["notebook_sources/common.py", "notebook_sources/pretrained_classifier.py",
         "scripts/run_classifier_experiment.py", "reviews/2026-11-22-photo-tests.json",
         "reviews/2026-11-22-v2/summary.json"]
for relative in FILES:
    target = CODE / relative
    target.parent.mkdir(parents=True, exist_ok=True)
    urllib.request.urlretrieve(
        f"https://raw.githubusercontent.com/chrisstayte/DominoesCalculator/{REVISION}/machine_learning/{relative}", target)
print("Pinned experiment code ready. Detector: domino-v2-01, unchanged.")

## Train and compare
Mount the existing Drive project. Each stage runs in a separate process to release
GPU memory between classifiers. Detailed outputs stay in the new private report
folder. Both candidates are trained before the development photos are compared.
Validation loss chooses each checkpoint; challenge-photo results do not choose it.

In [ ]:
from google.colab import drive
drive.mount("/content/drive")
if not PROJECT.is_dir():
    raise FileNotFoundError("The existing DominoesCalculator Drive project is required.")
JOB = PROJECT / "training_reports" / (COMPARISON + "-job")
JOB.mkdir(parents=True, exist_ok=False)
stages = [(name, ["train", "--run", name, "--variant", variant]) for name, variant in TRIALS]
stages.append(("compare", ["compare", "--runs", *[name for name, _ in TRIALS], "--comparison", COMPARISON]))
status = {"completed": [], "current": "starting", "revision": REVISION}
for name, arguments in stages:
    status["current"] = name
    (JOB / "status.json").write_text(json.dumps(status, indent=2) + "\n")
    print("STARTING", name, flush=True)
    command = [sys.executable, "-u", str(CODE / "scripts/run_classifier_experiment.py"),
               *arguments, "--project", str(PROJECT)]
    environment = dict(os.environ, MPLBACKEND="Agg", TF_CPP_MIN_LOG_LEVEL="2",
                       PYTHONUNBUFFERED="1", MPLCONFIGDIR="/content/domino-matplotlib")
    with (JOB / (name + ".log")).open("w") as log:
        process = subprocess.Popen(command, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                                   text=True, env=environment)
        for line in process.stdout:
            log.write(line)
            log.flush()
            print(line, end="", flush=True)
        code = process.wait()
    if code:
        status["failed"] = name
        (JOB / "status.json").write_text(json.dumps(status, indent=2) + "\n")
        raise RuntimeError(f"Stage {name} stopped; its private log and checkpoints are preserved.")
    status["completed"].append(name)
status["current"] = "complete"
(JOB / "status.json").write_text(json.dumps(status, indent=2) + "\n")
print("ALL CLASSIFIER EXPERIMENTS AND COMPARISON COMPLETE", flush=True)

## Results
This chart uses the same 36 selected development pairs for every classifier.
These are not independent full-photo accuracy estimates. The fixed detector still
contributes its existing missed/extra boxes. Review the private overlays as well.

In [ ]:
from IPython.display import Image, display
summary = json.loads((PROJECT / "training_reports" / COMPARISON / "summary.json").read_text())
for run in summary["runs"]:
    print(run["run"], "validation:", round(run["validation"]["original"]["accuracy"] * 100, 2), "%",
          "fixed original pairs:", run["fixed_original_crops"]["correct_pairs"], "/36",
          "frozen detector pairs:", run["frozen_detector_crops"]["correct_pairs"], "/36")
print("Protected inputs unchanged:", summary["protected_inputs_unchanged"])
display(Image(filename=str(PROJECT / "training_reports" / COMPARISON / "comparison.png")))